### Setup Environment

In [1]:
# Import libraries
import os
import opensmile
import pandas as pd
import librosa
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

import warnings
warnings.filterwarnings("ignore")

In [2]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


### Define Directories and Feature Sets

In [3]:
import os
import opensmile
import librosa
import numpy as np
import pandas as pd
import parselmouth
from tqdm import tqdm

class AudioFeatureExtractor:
    def __init__(self):
        # Initialize OpenSMILE
        self.feature_sets = [
            'ComParE_2016', 'eGeMAPSv01a', 'eGeMAPSv01b', 'eGeMAPSv02',
            'emobase', 'GeMAPSv01a', 'GeMAPSv01b'
        ]
        self.feature_levels = ['Functionals', 'LowLevelDescriptors']
        
        self.smile_instances = []
        for feature_set in self.feature_sets:
            for level in self.feature_levels:
                self.smile_instances.append(
                    opensmile.Smile(
                        feature_set=getattr(opensmile.FeatureSet, feature_set),
                        feature_level=getattr(opensmile.FeatureLevel, level)
                    )
                )
        
        # Define prefixes
        self.OPENSMILE_PREFIX = "os_"
        self.LIBROSA_PREFIX = "lb_"
        self.PRAAT_PREFIX = "pr_"

    def extract_opensmile_features(self, file_path):
        """Extract OpenSMILE features with prefix"""
        opensmile_features = {}
        for smile in self.smile_instances:
            try:
                features = smile.process_file(file_path)
                # Add prefix to each feature
                for key, value in features.to_dict(orient='records')[0].items():
                    opensmile_features[f"{self.OPENSMILE_PREFIX}{key}"] = value
            except Exception as e:
                print(f"OpenSMILE error for {file_path}: {e}")
        return opensmile_features

    def extract_librosa_features(self, file_path):
        """Extract Librosa features with prefix"""
        try:
            y, sr = librosa.load(file_path, sr=None)
            features = {
                "mfcc_mean": np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13), axis=1),
                "mfcc_std": np.std(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13), axis=1),
                "rolloff_mean": np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr)),
                "rolloff_std": np.std(librosa.feature.spectral_rolloff(y=y, sr=sr)),
                "zcr_mean": np.mean(librosa.feature.zero_crossing_rate(y)),
                "zcr_std": np.std(librosa.feature.zero_crossing_rate(y)),
                "centroid_mean": np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)),
                "centroid_std": np.std(librosa.feature.spectral_centroid(y=y, sr=sr)),
            }
            
            # Add prefix to features
            return {f"{self.LIBROSA_PREFIX}{k}": v for k, v in features.items()}
        except Exception as e:
            print(f"Librosa error for {file_path}: {e}")
            return {}

    def extract_praat_features(self, file_path):
        """Extract Praat features with prefix"""
        try:
            # Load the audio file using Parselmouth
            sound = parselmouth.Sound(file_path)

            # Initialize dictionary for Praat features
            praat_features = {}

            # Extract jitter and shimmer (via Praat commands)
            point_process = parselmouth.praat.call(sound, "To PointProcess (periodic, cc)", 75, 500)
            praat_features["jitter_local"] = parselmouth.praat.call(
                point_process, "Get jitter (local)", 0.0001, 0.02, 1.3, 1.6
            )
            praat_features["jitter_ddp"] = parselmouth.praat.call(
                point_process, "Get jitter (ddp)", 0.0001, 0.02, 1.3, 1.6
            )
            praat_features["shimmer_local"] = parselmouth.praat.call(
                [sound, point_process], "Get shimmer (local)", 0.0001, 0.02, 1.3, 1.6, 0.05, 1.6
            )
            praat_features["shimmer_apq3"] = parselmouth.praat.call(
                [sound, point_process], "Get shimmer (apq3)", 0.0001, 0.02, 1.3, 1.6, 0.05, 1.6
            )
            praat_features["shimmer_apq5"] = parselmouth.praat.call(
                [sound, point_process], "Get shimmer (apq5)", 0.0001, 0.02, 1.3, 1.6, 0.05, 1.6
            )

            # Extract Harmonics-to-Noise Ratio (HNR)
            praat_features["hnr"] = parselmouth.praat.call(sound, "Get HNR", 0.1, 0.25)

            # Extract formant frequencies using Burg's method
            formant = sound.to_formant_burg(time_step=0.01, max_number_of_formants=5, maximum_formant=5500)
            praat_features["formant1_mean"] = parselmouth.praat.call(formant, "Get mean", 1, 0, 0)
            praat_features["formant2_mean"] = parselmouth.praat.call(formant, "Get mean", 2, 0, 0)
            praat_features["formant3_mean"] = parselmouth.praat.call(formant, "Get mean", 3, 0, 0)
            praat_features["formant1_std"] = parselmouth.praat.call(formant, "Get standard deviation", 1, 0, 0)
            praat_features["formant2_std"] = parselmouth.praat.call(formant, "Get standard deviation", 2, 0, 0)
            praat_features["formant3_std"] = parselmouth.praat.call(formant, "Get standard deviation", 3, 0, 0)

            # Add prefix to features
            return {f"{self.PRAAT_PREFIX}{k}": v for k, v in praat_features.items()}

        except Exception as e:
            print(f"Error extracting Praat features for {file_path}: {e}")
            return {}

    # [Keep the rest of the AudioFeatureExtractor class methods the same]

    def extract_all_features(self, file_path):
        """Extract all features with prefixes"""
        features = {}
        
        # Extract features from each library
        opensmile_features = self.extract_opensmile_features(file_path)
        librosa_features = self.extract_librosa_features(file_path)
        praat_features = self.extract_praat_features(file_path)
        
        # Combine all features
        features.update(opensmile_features)
        features.update(librosa_features)
        features.update(praat_features)
        
        # Add feature counts
        features['opensmile_count'] = len(opensmile_features)
        features['librosa_count'] = len(librosa_features)
        features['praat_count'] = len(praat_features)
        features['total_features'] = len(features)
        
        return features

### Extract Features for All Feature Sets and Levels

In [4]:
def process_directory(directory, label, extractor):
    """Process all files in a directory"""
    feature_data = []
    wav_files = [f for f in os.listdir(directory) if f.endswith('.wav')]
    
    for file_name in tqdm(wav_files, desc=f"Processing {os.path.basename(directory)}"):
        file_path = os.path.join(directory, file_name)
        try:
            features = extractor.extract_all_features(file_path)
            features["file_name"] = file_name
            features["label"] = label
            feature_data.append(features)
        except Exception as e:
            print(f"Error processing {file_name}: {e}")
    
    return feature_data

In [ ]:
def main():
    # Initialize feature extractor
    extractor = AudioFeatureExtractor()
    
    # Define directories
    data_dir = r"C:\Users\sreev\OneDrive\Desktop\Parkinsons\Voice Samples for Patients with Parkinson's Disease and Healthy Controls_rd"
    healthy_dir = os.path.join(data_dir, "HC")
    parkinson_dir = os.path.join(data_dir, "PD")
    
    # Initialize feature data list
    feature_data = []
    
    # Process healthy controls
    print("Processing healthy controls...")
    healthy_data = process_directory(healthy_dir, label=0, extractor=extractor)
    feature_data.extend(healthy_data)
    
    # Process Parkinson's patients
    print("\nProcessing Parkinson's patients...")
    parkinson_data = process_directory(parkinson_dir, label=1, extractor=extractor)
    feature_data.extend(parkinson_data)
    
    # Convert to DataFrame and save
    features_df = pd.DataFrame(feature_data)
    output_file = "combined_features_081.csv"
    features_df.to_csv(output_file, index=False)
    
    # Print summary
    print(f"\nFeature extraction complete. Features saved to {output_file}")
    print("\nExtraction Summary:")
    print(f"Total files processed: {len(features_df)}")
    print(f"Healthy controls: {len(features_df[features_df['label'] == 0])}")
    print(f"Parkinson's patients: {len(features_df[features_df['label'] == 1])}")
    if len(features_df) > 0:
        print(f"Total features per file: {features_df['total_features'].iloc[0]}")
        print("\nFeature counts:")
        print(f"OpenSMILE features: {features_df['opensmile_count'].iloc[0]}")
        print(f"Librosa features: {features_df['librosa_count'].iloc[0]}")
        print(f"Praat features: {features_df['praat_count'].iloc[0]}")

if __name__ == "__main__":
    main()

Processing healthy controls...


Processing HC:   5%|▍         | 1/22 [01:43<36:22, 103.94s/it]

Error extracting Praat features for C:\Users\sreev\OneDrive\Desktop\Parkinsons\Voice Samples for Patients with Parkinson's Disease and Healthy Controls_rd\HC\ID00_hc_0_0_0.wav: Command requires more than the given 4 arguments: no value for argument "Maximum period factor".


In [5]:
dataset = pd.read_csv("combined_features_081.csv")

# Display the first few rows
print(dataset.head())

# Display basic info about the dataset
print(dataset.info())

# Summary statistics of numerical columns
print(dataset.describe())


   os_audspec_lengthL1norm_sma_range  os_audspec_lengthL1norm_sma_maxPos  \
0                           1.068221                            0.229012   
1                           6.369210                            0.493568   
2                           1.439469                            0.892386   
3                           1.419298                            0.484889   
4                           2.324570                            0.133192   

   os_audspec_lengthL1norm_sma_minPos  os_audspec_lengthL1norm_sma_quartile1  \
0                            0.053761                               0.070124   
1                            0.760166                               0.081711   
2                            0.905793                               0.089911   
3                            0.700226                               0.068416   
4                            0.444155                               0.074566   

   os_audspec_lengthL1norm_sma_quartile2  \
0                 

In [6]:
df=dataset[:]

In [7]:
df.drop("file_name", axis=1)

,os_audspec_lengthL1norm_sma_range,os_audspec_lengthL1norm_sma_maxPos,os_audspec_lengthL1norm_sma_minPos,os_audspec_lengthL1norm_sma_quartile1,os_audspec_lengthL1norm_sma_quartile2,os_audspec_lengthL1norm_sma_quartile3,os_audspec_lengthL1norm_sma_iqr1-2,os_audspec_lengthL1norm_sma_iqr2-3,os_audspec_lengthL1norm_sma_iqr1-3,os_audspec_lengthL1norm_sma_percentile1.0,...,lb_rolloff_std,lb_zcr_mean,lb_zcr_std,lb_centroid_mean,lb_centroid_std,opensmile_count,librosa_count,praat_count,total_features,label
0,1.068221,0.229012,0.053761,0.070124,0.142961,0.253994,0.072837,0.111032,0.183870,0.041453,...,5199.429991,0.078671,0.090998,4145.632542,2330.139801,7188,8,0,7199,0
1,6.369210,0.493568,0.760166,0.081711,0.370981,0.694179,0.289270,0.323198,0.612468,0.039616,...,4349.104197,0.096239,0.105654,4914.428175,2458.087281,7188,8,0,7199,0
2,1.439469,0.892386,0.905793,0.089911,0.195657,0.352913,0.105746,0.157256,0.263002,0.042567,...,5377.237335,0.086954,0.090219,4144.127362,2432.865223,7188,8,0,7199,0
3,1.419298,0.484889,0.700226,0.068416,0.244612,0.444998,0.176196,0.200386,0.376582,0.041061,...,5555.861414,0.089911,0.100396,4473.216471,2631.507539,7188,8,0,7199,0
4,2.324570,0.133192,0.444155,0.074566,0.182165,0.314206,0.107599,0.132041,0.239640,0.035791,...,5130.270252,0.096233,0.104851,4454.264244,2461.130183,7188,8,0,7199,0
5,1.870663,0.141388,0.519458,0.061608,0.196951,0.365395,0.135344,0.168444,0.303787,0.032335,...,4896.742224,0.103472,0.111162,5164.314146,2639.562889,7188,8,0,7199,0
6,2.085477,0.906291,0.878493,0.079178,0.212332,0.413533,0.133154,0.201201,0.334355,0.044516,...,5354.603217,0.088560,0.105497,4457.359932,2657.107240,7188,8,0,7199,0
7,0.883342,0.166273,0.055424,0.043023,0.115201,0.241436,0.072178,0.126236,0.198414,0.029963,...,5510.066722,0.091658,0.103316,4932.043581,2602.458446,7188,8,0,7199,0
8,1.780565,0.709106,0.198835,0.064453,0.175379,0.383468,0.110925,0.208090,0.319015,0.034262,...,5255.977184,0.089864,0.100429,4502.395682,2469.954110,7188,8,0,7199,0
9,0.932374,0.743614,0.346798,0.053095,0.123460,0.217827,0.070365,0.094367,0.164733,0.033745,...,4802.823200,0.101355,0.109156,5115.553527,2606.626353,7188,8,0,7199,0


In [8]:
import re
import ast
import numpy as np

# Step 1: Fix formatting in MFCC columns (now with 'lb_' prefix)
def fix_format(value):
    if isinstance(value, str):
        fixed_value = re.sub(r'(?<=\d)\s+(?=[-\d])', ', ', value)
        try:
            return ast.literal_eval(fixed_value)
        except:
            return None
    return value

# Apply fix_format to prefixed MFCC columns
df['lb_mfcc_mean'] = df['lb_mfcc_mean'].apply(fix_format)
df['lb_mfcc_std'] = df['lb_mfcc_std'].apply(fix_format)

# Step 2: Compute global statistics for 'lb_mfcc_mean'
df['lb_global_mfcc_mean'] = df['lb_mfcc_mean'].apply(lambda x: np.mean(x) if x is not None else None)
df['lb_global_mfcc_mean_std'] = df['lb_mfcc_mean'].apply(lambda x: np.std(x) if x is not None else None)
df['lb_global_mfcc_mean_min'] = df['lb_mfcc_mean'].apply(lambda x: np.min(x) if x is not None else None)
df['lb_global_mfcc_mean_max'] = df['lb_mfcc_mean'].apply(lambda x: np.max(x) if x is not None else None)

# Step 3: Compute global statistics for 'lb_mfcc_std'
df['lb_global_mfcc_std_mean'] = df['lb_mfcc_std'].apply(lambda x: np.mean(x) if x is not None else None)
df['lb_global_mfcc_std_std'] = df['lb_mfcc_std'].apply(lambda x: np.std(x) if x is not None else None)
df['lb_global_mfcc_std_min'] = df['lb_mfcc_std'].apply(lambda x: np.min(x) if x is not None else None)
df['lb_global_mfcc_std_max'] = df['lb_mfcc_std'].apply(lambda x: np.max(x) if x is not None else None)

# Step 4: Drop the original list-based columns
df = df.drop(columns=['lb_mfcc_mean', 'lb_mfcc_std'])

# Save the processed dataset
df.to_csv("processed_features_for_ml_081.csv", index=False)
print("Global statistics calculated and dataset saved as 'processed_features_for_ml_081.csv'.")

Global statistics calculated and dataset saved as 'processed_features_for_ml_081.csv'.


**Column Names**

In [9]:
df.columns

Index(['os_audspec_lengthL1norm_sma_range',
       'os_audspec_lengthL1norm_sma_maxPos',
       'os_audspec_lengthL1norm_sma_minPos',
       'os_audspec_lengthL1norm_sma_quartile1',
       'os_audspec_lengthL1norm_sma_quartile2',
       'os_audspec_lengthL1norm_sma_quartile3',
       'os_audspec_lengthL1norm_sma_iqr1-2',
       'os_audspec_lengthL1norm_sma_iqr2-3',
       'os_audspec_lengthL1norm_sma_iqr1-3',
       'os_audspec_lengthL1norm_sma_percentile1.0',
       ...
       'file_name', 'label', 'lb_global_mfcc_mean', 'lb_global_mfcc_mean_std',
       'lb_global_mfcc_mean_min', 'lb_global_mfcc_mean_max',
       'lb_global_mfcc_std_mean', 'lb_global_mfcc_std_std',
       'lb_global_mfcc_std_min', 'lb_global_mfcc_std_max'],
      dtype='object', length=7208)

In [10]:
df=df.drop("file_name", axis=1)

In [11]:
correlations = df.corrwith(df['label']).sort_values(ascending=False)
print("Feature Correlations with Label:\n", correlations)


Feature Correlations with Label:
 label                                                     1.000000
os_pcm_fftMag_spectralSkewness_sma_percentile1.0          0.748721
os_pcm_fftMag_spectralRollOff50.0_sma_de_percentile1.0    0.683342
os_audSpec_Rfilt_sma[8]_minRangeRel                       0.682789
os_pcm_fftMag_spectralRollOff25.0_sma_de_percentile1.0    0.665361
                                                            ...   
os_F0env_sma_de_quartile2                                      NaN
opensmile_count                                                NaN
librosa_count                                                  NaN
praat_count                                                    NaN
total_features                                                 NaN
Length: 7207, dtype: float64


In [12]:
correlation_matrix = df.corr()

# Correlation of features with the label
label_correlation = correlation_matrix['label']

# Step 2: Find highly correlated pairs (absolute correlation > 0.9)
threshold = 0.9
high_corr_pairs = []

for i in range(correlation_matrix.shape[0]):
    for j in range(i + 1, correlation_matrix.shape[1]):  # Avoid self-correlation
        if abs(correlation_matrix.iloc[i, j]) > threshold:
            high_corr_pairs.append((correlation_matrix.index[i], correlation_matrix.columns[j], correlation_matrix.iloc[i, j]))

# Display highly correlated pairs
high_corr_pairs = sorted(high_corr_pairs, key=lambda x: -abs(x[2]))  # Sort by correlation value
for pair in high_corr_pairs:
    print(f"{pair[0]} and {pair[1]}: Correlation = {pair[2]:.2f}")

os_F0env_sma_min and os_F0_sma: Correlation = 1.00
os_pcm_fftMag_spectralRollOff25.0_sma_percentile99.0 and os_pcm_fftMag_spectralRollOff25.0_sma_pctlrange0-1: Correlation = 1.00
os_pcm_fftMag_spectralEntropy_sma_minSegLen and os_mfcc_sma[1]_minSegLen: Correlation = 1.00
os_F0final_sma_amean and os_F0final_sma_posamean: Correlation = 1.00
os_voicingFinalUnclipped_sma_amean and os_voicingFinalUnclipped_sma_posamean: Correlation = 1.00
os_jitterLocal_sma_amean and os_jitterLocal_sma_posamean: Correlation = 1.00
os_jitterDDP_sma_amean and os_jitterDDP_sma_posamean: Correlation = 1.00
os_shimmerLocal_sma_amean and os_shimmerLocal_sma_posamean: Correlation = 1.00
os_pcm_fftMag_spectralCentroid_sma_de_peakMeanRel and os_pcm_fftMag_psySharpness_sma_de_peakMeanRel: Correlation = 1.00
os_audspec_lengthL1norm_sma and os_Loudness_sma3: Correlation = 1.00
os_F0_sma_max and os_F0_sma_range: Correlation = 1.00
os_F0_sma_quartile2 and os_F0_sma_iqr1-2: Correlation = 1.00
os_F0_sma_quartile3 and os_F0

In [13]:
features_to_drop = set()

for feature1, feature2, _ in high_corr_pairs:  # Unpack the correlation value as `_`
    # Compare correlation with the label
    if abs(label_correlation[feature1]) > abs(label_correlation[feature2]):
        features_to_drop.add(feature2)  # Drop feature2
    else:
        features_to_drop.add(feature1)  # Drop feature1

# Step 4: Drop the selected features from the dataset
df_reduced = df.drop(columns=features_to_drop)

print(f"Features dropped: {features_to_drop}")
print(f"Reduced dataset now has {df_reduced.shape[1]} features.")

Features dropped: {'os_audSpec_Rfilt_sma[23]_meanRisingSlope', 'os_audSpec_Rfilt_sma_de[10]_meanPeakDist', 'os_mfcc_sma_de[2]_min', 'os_pcm_intensity_sma_range', 'os_pcm_RMSenergy_sma_meanRisingSlope', 'os_audSpec_Rfilt_sma_de[16]_quartile3', 'os_audSpec_Rfilt_sma[2]_quartile3', 'os_lspFreq_sma[6]_quartile2', 'os_mfcc_sma_de[2]_lpgain', 'os_pcm_fftMag_spectralFlux_sma_segLenStddev', 'os_pcm_zcr_sma_lpc1', 'os_audSpec_Rfilt_sma[20]_qregc2', 'os_lspFreq_sma[5]_iqr1-3', 'os_pcm_zcr_sma_meanRisingSlope', 'os_pcm_fftMag_spectralCentroid_sma_minSegLen', 'os_audSpec_Rfilt_sma_de[11]_maxPos', 'os_audSpec_Rfilt_sma_de[2]_iqr1-3', 'os_audSpec_Rfilt_sma[19]_upleveltime50', 'os_audSpec_Rfilt_sma_de[0]_meanRisingSlope', 'os_mfcc_sma[6]_iqr1-2', 'os_audSpec_Rfilt_sma[16]_risetime', 'os_pcm_fftMag_spectralRollOff25.0_sma_de_lpc0', 'os_audSpec_Rfilt_sma[6]_amean', 'os_audSpec_Rfilt_sma[2]_stddev', 'os_audSpec_Rfilt_sma_de[24]_stddevFallingSlope', 'os_audSpec_Rfilt_sma_de[5]_peakMeanAbs', 'os_mfcc_sma[

In [14]:
df_reduced.to_csv("reduced_features_for_ml_081.csv", index=False)
print("Reduced dataset saved as 'reduced_features_for_ml_081.csv'")


Reduced dataset saved as 'reduced_features_for_ml_081.csv'


In [19]:
import pandas as pd

def safe_select_columns(df, desired_columns):
    """
    Safely select columns from a DataFrame, handling missing columns gracefully.
    """
    # Find which columns exist and which are missing
    existing_columns = [col for col in desired_columns if col in df.columns]
    missing_columns = [col for col in desired_columns if col not in df.columns]
    
    # Print information about found and missing columns
    print(f"\nFound {len(existing_columns)} columns out of {len(desired_columns)} requested")
    if missing_columns:
        print(f"\nWarning: {len(missing_columns)} columns were not found in the DataFrame:")
        for col in missing_columns:
            print(f"- {col}")
    
    # Create new DataFrame with only existing columns
    return df[existing_columns].copy()

# Your list of columns to select
columns_to_select = [
    'os_audspec_lengthL1norm_sma_upleveltime50', 'total_features', 'praat_count',
    'os_pcm_fftMag_spectralSlope_sma_upleveltime90', 'os_audSpec_Rfilt_sma_de_4__maxPos',
    'os_audSpec_Rfilt_sma_25__upleveltime25', 'os_pcm_zcr_sma_min',
    'os_audspec_lengthL1norm_sma_leftctime', 'os_audSpec_Rfilt_sma_24__quartile2',
    'lb_zcr_mean', 'os_audspec_lengthL1norm_sma_kurtosis',
    'os_audSpec_Rfilt_sma_de_23__maxPos', 'lb_global_mfcc_mean',
    'os_audSpec_Rfilt_sma_de_25__lpgain', 'os_voiceProb_sma_linregc1',
    'os_audspec_lengthL1norm_sma_lpc3', 'os_audspec_lengthL1norm_sma_maxSegLen',
    'os_pcm_zcr_sma_linregc2', 'os_pcm_zcr_sma', 'os_pcm_zcr_sma_de_skewness',
    'os_audspec_lengthL1norm_sma_meanSegLen', 'os_pcm_fftMag_spectralVariance_sma_range',
    'os_audspecRasta_lengthL1norm_sma_maxPos', 'os_audspec_lengthL1norm_sma_minSegLen',
    'os_slope500-1500_sma3', 'opensmile_count', 'os_audspec_lengthL1norm_sma_iqr2-3',
    'os_pcm_fftMag_spectralEntropy_sma_qregc2', 'os_pcm_loudness_sma_minPos',
    'os_F0_sma_linregc1', 'os_audspec_lengthL1norm_sma_upleveltime25',
    'os_logHNR_sma_de_lpc1', 'os_pcm_fftMag_spectralEntropy_sma_de_minPos',
    'os_audspec_lengthL1norm_sma_percentile1.0', 'os_audSpec_Rfilt_sma_de_1__maxPos',
    'librosa_count', 'os_audspec_lengthL1norm_sma_iqr1-2', 'os_pcm_zcr_sma_maxPos',
    'os_audspec_lengthL1norm_sma_upleveltime90', 'os_audspec_lengthL1norm_sma_maxPos',
    'lb_global_mfcc_mean_min', 'os_pcm_zcr_sma_de_linregc1',
    'os_pcm_fftMag_spectralSlope_sma_flatness', 'os_pcm_loudness_sma_de_amean',
    'os_audspec_lengthL1norm_sma_lpc2', 'os_audspec_lengthL1norm_sma_risetime',
    'os_audspec_lengthL1norm_sma_lpgain', 'os_audSpec_Rfilt_sma_de_21__lpgain',
    'os_audspec_lengthL1norm_sma_upleveltime75', 'os_audspec_lengthL1norm_sma_lpc0',
    'os_pcm_zcr_sma_de_linregc2', 'os_audspec_lengthL1norm_sma_skewness', 'label'
]

# Print original shape
print("Original DataFrame shape:", df_reduced.shape)

# Print available columns for debugging
print("\nAvailable columns in DataFrame:")
print(df_reduced.columns.tolist())

# Safely select columns
df_reduced = safe_select_columns(df_reduced, columns_to_select)

# Print final shape
print("\nFinal DataFrame shape:", df_reduced.shape)

# Save to CSV
df_reduced.to_csv("df_reduced_081.csv", index=False)
print("\nSaved DataFrame to df_reduced_081.csv")

Original DataFrame shape: (38, 3357)

Available columns in DataFrame:
['os_audspec_lengthL1norm_sma_maxPos', 'os_audspec_lengthL1norm_sma_minPos', 'os_audspec_lengthL1norm_sma_quartile1', 'os_audspec_lengthL1norm_sma_kurtosis', 'os_audspec_lengthL1norm_sma_meanSegLen', 'os_audspec_lengthL1norm_sma_maxSegLen', 'os_audspec_lengthL1norm_sma_segLenStddev', 'os_audspec_lengthL1norm_sma_upleveltime25', 'os_audspec_lengthL1norm_sma_upleveltime75', 'os_audspec_lengthL1norm_sma_upleveltime90', 'os_audspec_lengthL1norm_sma_leftctime', 'os_audspec_lengthL1norm_sma_lpc1', 'os_audspec_lengthL1norm_sma_lpc4', 'os_audspecRasta_lengthL1norm_sma_maxPos', 'os_audspecRasta_lengthL1norm_sma_minPos', 'os_audspecRasta_lengthL1norm_sma_quartile1', 'os_audspecRasta_lengthL1norm_sma_quartile2', 'os_audspecRasta_lengthL1norm_sma_percentile1.0', 'os_audspecRasta_lengthL1norm_sma_skewness', 'os_audspecRasta_lengthL1norm_sma_meanSegLen', 'os_audspecRasta_lengthL1norm_sma_maxSegLen', 'os_audspecRasta_lengthL1norm_s

In [18]:
c=['os_audspec_lengthL1norm_sma_upleveltime50', 'os_audSpec_Rfilt_sma_de_4__maxPos', 'os_audSpec_Rfilt_sma_25__upleveltime25', 'os_audSpec_Rfilt_sma_24__quartile2', 'lb_zcr_mean', 'os_audSpec_Rfilt_sma_de_23__maxPos', 'os_audSpec_Rfilt_sma_de_25__lpgain', 'os_audspec_lengthL1norm_sma_lpc3', 'os_pcm_zcr_sma', 'os_pcm_fftMag_spectralVariance_sma_range', 'os_audspec_lengthL1norm_sma_minSegLen', 'os_audspec_lengthL1norm_sma_iqr2-3', 'os_pcm_fftMag_spectralEntropy_sma_qregc2', 'os_audspec_lengthL1norm_sma_percentile1.0', 'os_audSpec_Rfilt_sma_de_1__maxPos', 'os_audspec_lengthL1norm_sma_iqr1-2', 'lb_global_mfcc_mean_min', 'os_pcm_zcr_sma_de_linregc1', 'os_pcm_fftMag_spectralSlope_sma_flatness', 'os_pcm_loudness_sma_de_amean', 'os_audspec_lengthL1norm_sma_lpc2', 'os_audspec_lengthL1norm_sma_risetime', 'os_audspec_lengthL1norm_sma_lpgain', 'os_audSpec_Rfilt_sma_de_21__lpgain', 'os_audspec_lengthL1norm_sma_lpc0', 'os_pcm_zcr_sma_de_linregc2', 'os_audspec_lengthL1norm_sma_skewness']
j=0
for i in c:
    j+=1
print(j)

27


In [20]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Separate features and target
X = df_reduced.drop(columns=['label'])  # Features
y = df_reduced['label']  # Target

# Choose a scaler
scaler = StandardScaler()  # Use MinMaxScaler() for min-max scaling

# Fit and transform the features
X_scaled = scaler.fit_transform(X)

# Convert back to a DataFrame for easier handling
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

# Combine scaled features with the target variable
df_scaled = pd.concat([X_scaled_df, y.reset_index(drop=True)], axis=1)

# Save the scaled dataset
df_scaled.to_csv("scaled_features_for_ml_081.csv", index=False)

print("Features scaled and saved as 'scaled_features_for_ml_081.csv'.")


Features scaled and saved as 'scaled_features_for_ml_081.csv'.


In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
import pandas as pd

# Load scaled dataset
df_scaled = pd.read_csv("scaled_features_for_ml_081.csv")

# Separate features and target
X = df_scaled.drop(columns=['label'])  # Features
y = df_scaled['label']  # Target

# Split the dataset into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Apply SMOTE to the training set
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# Check the new class distribution
print("Class distribution before SMOTE:", y_train.value_counts())
print("Class distribution after SMOTE:", pd.Series(y_train_res).value_counts())

# Save the resampled training data
X_train_res_df = pd.DataFrame(X_train_res, columns=X_train.columns)
y_train_res_df = pd.DataFrame(y_train_res, columns=['label'])

# Combine resampled features and target into a single DataFrame
train_resampled = pd.concat([X_train_res_df, y_train_res_df], axis=1)
train_resampled.to_csv("resampled_train_data_081.csv", index=False)

print("Resampled training data saved as 'resampled_train_data_081.csv'.")


Class distribution before SMOTE: label
0    17
1    13
Name: count, dtype: int64
Class distribution after SMOTE: label
0    17
1    17
Name: count, dtype: int64
Resampled training data saved as 'resampled_train_data_081.csv'.


: 

## Train Baseline Models

### SVM Model

In [76]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Train an SVM model
svm_model = SVC(kernel='linear', probability=True, random_state=42)
svm_model.fit(X_train_res, y_train_res)

# Evaluate the SVM model
y_pred_svm = svm_model.predict(X_test)

# Print evaluation metrics
print(f"SVM Accuracy: {accuracy_score(y_test, y_pred_svm):.2f}")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_svm))
print("Classification Report:\n", classification_report(y_test, y_pred_svm))


SVM Accuracy: 0.71
Confusion Matrix:
 [[8 1]
 [4 4]]
Classification Report:
               precision    recall  f1-score   support

           0       0.67      0.89      0.76         9
           1       0.80      0.50      0.62         8

    accuracy                           0.71        17
   macro avg       0.73      0.69      0.69        17
weighted avg       0.73      0.71      0.69        17



### Random Forest Model

In [78]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Train a Random Forest model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_res, y_train_res)

# Evaluate the Random Forest model
y_pred_rf = rf_model.predict(X_test)

# Print evaluation metrics
print(f"Random Forest Accuracy: {accuracy_score(y_test, y_pred_rf):.2f}")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("Classification Report:\n", classification_report(y_test, y_pred_rf))


Random Forest Accuracy: 0.71
Confusion Matrix:
 [[8 1]
 [4 4]]
Classification Report:
               precision    recall  f1-score   support

           0       0.67      0.89      0.76         9
           1       0.80      0.50      0.62         8

    accuracy                           0.71        17
   macro avg       0.73      0.69      0.69        17
weighted avg       0.73      0.71      0.69        17



### MLP Model

In [89]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Define the baseline MLP model
mlp_model = Sequential([
    Dense(64, input_dim=X_train_res.shape[1], activation='relu'),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')  # Output layer for binary classification
])

# Compile the model
mlp_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
mlp_history = mlp_model.fit(X_train_res, y_train_res, epochs=30, batch_size=16, validation_data=(X_test, y_test), verbose=1)

# Evaluate the model
mlp_loss, mlp_accuracy = mlp_model.evaluate(X_test, y_test)
print(f"Baseline MLP Accuracy: {mlp_accuracy:.2f}")


Epoch 1/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 83ms/step - accuracy: 0.7021 - loss: 0.6345 - val_accuracy: 0.8235 - val_loss: 0.3913
Epoch 2/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.9833 - loss: 0.0777 - val_accuracy: 0.8235 - val_loss: 0.4719
Epoch 3/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 1.0000 - loss: 0.0218 - val_accuracy: 0.8235 - val_loss: 0.5195
Epoch 4/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 1.0000 - loss: 0.0071 - val_accuracy: 0.8235 - val_loss: 0.5480
Epoch 5/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 1.0000 - loss: 0.0039 - val_accuracy: 0.8235 - val_loss: 0.5629
Epoch 6/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 1.0000 - loss: 0.0019 - val_accuracy: 0.8235 - val_loss: 0.5708
Epoch 7/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 1.0000 - loss: 0.0011 - val_accuracy: 0.8235 - val_loss: 0.5748
Epoch 8/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 1.0000 - loss: 0.0011 - val_accuracy: 0.8235 - val_loss: 0.5774


### CNN Model

In [92]:
# Ensure the data is in NumPy array format
X_train_res = X_train_res.values if isinstance(X_train_res, pd.DataFrame) else X_train_res
X_test = X_test.values if isinstance(X_test, pd.DataFrame) else X_test

# Reshape the data
X_train = X_train_res.reshape(X_train_res.shape[0], X_train_res.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")


X_train shape: (64, 4295, 1)
X_test shape: (17, 4295, 1)


In [93]:
from tensorflow.keras.utils import to_categorical

y_train = to_categorical(y_train_res, num_classes=2)
y_test = to_categorical(y_test, num_classes=2)


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
import numpy as np

# Assuming you have your scaled dataset ready
X = df_scaled.drop(columns=['label']).values  # Features
y = df_scaled['label'].values  # Target

# Convert the target to categorical (for multi-class classification)
y = to_categorical(y, num_classes=2)  # Use num_classes=N for N-class problems

# Reshape X for CNN + RNN input (samples, timesteps, features)
# Assuming the time dimension is inherent in the features, reshape accordingly
X = X.reshape(X.shape[0], X.shape[1], 1)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the CNN + RNN model
model = Sequential()

# CNN layers
model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(X.shape[1], 1)))
model.add(MaxPooling1D(pool_size=2))
model.add(Dropout(0.3))

# RNN layers
model.add(LSTM(100, activation="relu", return_sequences=True))
model.add(LSTM(100, activation="relu", return_sequences=True))
model.add(Dropout(0.1))
model.add(GRU(256, return_sequences=False))  # Note: Last RNN should return_sequences=False

# Fully connected layers
model.add(Dense(128, activation='relu'))
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(2, activation='sigmoid'))  # Change 2 to N for N-class problems

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Add EarlyStopping callback
early_stopping = EarlyStopping(
    monitor='val_loss',  # Monitor validation loss
    patience=10,         # Number of epochs to wait after no improvement
    restore_best_weights=True,  # Restore weights from the epoch with the best validation loss
    verbose=1
)

# Train the model
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=200,
    batch_size=32,
    verbose=1,
    callbacks=[early_stopping]  # Include EarlyStopping in callbacks
)

# Evaluate the model
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=1)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")


Epoch 1/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 14s 5s/step - accuracy: 0.4700 - loss: 0.6934 - val_accuracy: 0.4615 - val_loss: 0.6929
Epoch 2/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 10s 5s/step - accuracy: 0.4961 - loss: 0.6937 - val_accuracy: 0.4615 - val_loss: 0.6929
Epoch 3/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 10s 5s/step - accuracy: 0.5797 - loss: 0.6888 - val_accuracy: 0.4615 - val_loss: 0.6927
Epoch 4/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 12s 5s/step - accuracy: 0.6005 - loss: 0.6858 - val_accuracy: 0.4615 - val_loss: 0.6932
Epoch 5/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 11s 6s/step - accuracy: 0.5613 - loss: 0.6897 - val_accuracy: 0.4615 - val_loss: 0.6940
Epoch 6/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 19s 5s/step - accuracy: 0.5639 - loss: 0.6925 - val_accuracy: 0.4615 - val_loss: 0.6940
Epoch 7/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 9s 5s/step - accuracy: 0.5743 - loss: 0.6773 - val_accuracy: 0.4615 - val_loss: 0.6941
Epoch 8/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 10s 5s/step - accuracy: 0.5639 - loss: 0.6828 - val_accuracy: 0.4615 - val_loss: 0.6942
E

## Hyperparameter Tuning
### Grid Search for Random Forest

In [71]:
from sklearn.model_selection import GridSearchCV

# Define hyperparameter grid
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10]
}

# Perform Grid Search
grid_search = GridSearchCV(estimator=rf_model, param_grid=param_grid, cv=3, scoring='accuracy', verbose=1)
grid_search.fit(X_train, y_train)

# Best Parameters
print(f"Best Random Forest Parameters: {grid_search.best_params_}")

# Evaluate the best model
best_rf_model = grid_search.best_estimator_
y_pred_best_rf = best_rf_model.predict(X_test)
print(f"Random Forest Accuracy (Best Parameters): {accuracy_score(y_test, y_pred_best_rf):.2f}")


Fitting 3 folds for each of 36 candidates, totalling 108 fits
Best Random Forest Parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
Random Forest Accuracy (Best Parameters): 0.76


### Ensemble Learning

In [121]:
from sklearn.ensemble import VotingClassifier
from scikeras.wrappers import KerasClassifier

# Wrap MLP model
def create_model():
    model = Sequential([
        Dense(128, activation='relu', input_dim=X_train.shape[1]),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

mlp_sklearn = KerasClassifier(build_fn=create_model, epochs=50, batch_size=16, verbose=0)

# Ensemble Model
ensemble = VotingClassifier(estimators=[('svm', svm_model), ('rf', rf_model), ('mlp', mlp_sklearn)], voting='soft')
ensemble.fit(X_train, y_train)

# Evaluate Ensemble
y_pred_ensemble = ensemble.predict(X_test)
print(f"Ensemble Accuracy: {accuracy_score(y_test, y_pred_ensemble):.2f}")


Ensemble Accuracy: 0.76


### Enhanced CNN

In [119]:
def build_cnn_model(hp):
    model = Sequential()
    model.add(Conv1D(filters=hp.Int('filters_1', min_value=16, max_value=64, step=16),
                     kernel_size=hp.Choice('kernel_size', values=[3, 5]),
                     activation='relu', input_shape=(X_train.shape[1], 1)))
    model.add(MaxPooling1D(pool_size=hp.Choice('pool_size', values=[2, 3])))
    model.add(Flatten())
    model.add(Dense(hp.Int('dense_units', min_value=32, max_value=128, step=32), activation='relu'))
    model.add(Dropout(hp.Float('dropout', min_value=0.2, max_value=0.5, step=0.1)))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Tuning with Keras Tuner
cnn_tuner = kt.Hyperband(build_cnn_model, objective='val_accuracy', max_epochs=40, factor=3, directory='cnn_tuning', project_name='cnn')

cnn_tuner.search(X_train_cnn, y_train, epochs=50, validation_data=(X_test_cnn, y_test), verbose=1)

# Train the best CNN model
best_cnn_hps = cnn_tuner.get_best_hyperparameters(num_trials=1)[0]
best_cnn_model = cnn_tuner.hypermodel.build(best_cnn_hps)
best_cnn_model.fit(X_train_cnn, y_train, epochs=40, batch_size=16, validation_data=(X_test_cnn, y_test))

# Evaluate the best CNN model
cnn_loss, cnn_accuracy = best_cnn_model.evaluate(X_test_cnn, y_test)
print(f"Enhanced CNN Accuracy: {cnn_accuracy:.2f}")


Reloading Tuner from cnn_tuning\cnn\tuner0.json
Epoch 1/40
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 87ms/step - accuracy: 0.5316 - loss: 0.6849 - val_accuracy: 0.4000 - val_loss: 0.6900
Epoch 2/40
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6318 - loss: 0.6594 - val_accuracy: 0.3200 - val_loss: 0.6892
Epoch 3/40
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.6401 - loss: 0.6582 - val_accuracy: 0.3600 - val_loss: 0.6857
Epoch 4/40
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6318 - loss: 0.6311 - val_accuracy: 0.4800 - val_loss: 0.6823
Epoch 5/40
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7091 - loss: 0.6173 - val_accuracy: 0.4800 - val_loss: 0.6755
Epoch 6/40
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.6610 - loss: 0.6338 - val_accuracy: 0.6000 - val_loss: 0.6679
Epoch 7/40
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.6659 - loss: 0.5926 - val_accuracy: 0.6000 - val_loss: 0.6602
Epoch 8/40
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6390 - loss: 0